Installing necessary dependencies

In [44]:
%pip install -q chromadb pymupdf sentence-transformers tqdm python-dotenv
%pip install -q pandas rich
%pip install pymupdf


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Loading the policy PDF and reading the text

In [45]:
from __future__ import annotations
import re, json, os
from dataclasses import dataclass, asdict
from typing import List, Dict
import fitz  

# === 1) Configuration
PDF_PATH = "E:/Study/WORK/Join Venture Ai/For Task - Policy file.pdf"   
SAVE_JSONL_TO = "data/processed/pages.jsonl"  # optional: where to save per-page text for inspection

# === 2) Lightweight cleaners ===
HEADER_FOOTER_PATTERNS = [
    r"^\s*Page\s+\d+\s*$",          # "Page 12"
    r"^\s*\d+\s*$",                 # bare page number on a line
    r"^\s*Table\s+of\s+Contents.*$",# generic headings 
]

def _strip_headers_footers(text: str) -> str:
    """Remove common header/footer lines that often repeat on every page."""
    lines = text.splitlines()
    cleaned = []
    for ln in lines:
        if any(re.search(pat, ln, flags=re.IGNORECASE) for pat in HEADER_FOOTER_PATTERNS):
            continue
        cleaned.append(ln)
    return "\n".join(cleaned)

def _dehyphenate(text: str) -> str:
    """
    Join words split across line breaks with hyphens:
    e.g., 'gov-\nernment' -> 'government'
    """
    return re.sub(r"(\w)-\n(\w)", r"\1\2", text)

def _normalize_whitespace(text: str) -> str:
    """
    Collapse excessive whitespace but preserve paragraph breaks.
    """
    # Normalize newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Convert multiple blank lines to a single blank line
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse spaces on each line
    text = "\n".join(re.sub(r"[ \t]{2,}", " ", ln).strip() for ln in text.splitlines())
    # Remove leading/trailing blank lines
    return text.strip()

def basic_clean(text: str) -> str:
    """Pipeline of simple, safe cleanups."""
    text = _strip_headers_footers(text)
    text = _dehyphenate(text)
    text = _normalize_whitespace(text)
    return text

# === 3) Data structure ===
@dataclass
class PageText:
    page: int          # 1-based page index
    text: str          # cleaned text of that page

# === 4) Reader ===
def read_pdf_pages(pdf_path: str) -> List[PageText]:
    """
    Read a PDF with PyMuPDF, return a list of PageText(page, text) with light cleaning.
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found at: {pdf_path}")

    doc = fitz.open(pdf_path)
    pages: List[PageText] = []

    for i in range(len(doc)):  
        raw = doc[i].get_text("text") or ""
        cleaned = basic_clean(raw)
        pages.append(PageText(page=i + 1, text=cleaned))

    doc.close()
    return pages

# === 5) Save per-page JSONL for inspection ===
def save_pages_jsonl(pages: List[PageText], out_path: str) -> None:
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for p in pages:
            f.write(json.dumps(asdict(p), ensure_ascii=False) + "\n")

# === 6) Run Phase 1 ===
pages = read_pdf_pages(PDF_PATH)

print(f"Parsed pages: {len(pages)}")
print("\n----- Page 1 (first 700 chars) -----\n")
print(pages[0].text[:700])

# Optionally save per-page text for later inspection / debugging
save_pages_jsonl(pages, SAVE_JSONL_TO)
print(f"\nSaved page texts to: {SAVE_JSONL_TO}")


Parsed pages: 6

----- Page 1 (first 700 chars) -----

1.2
FINANCIAL POLICY OBJECTIVES AND STRATEGIES

STATEMENT
The presentation and preparation of the Territory’s Budget is provided for in sections 11 and
11A of the Financial Management Act 1996 (the Act).
The purpose of the financial policy objectives and strategies statement is to make transparent
the Government’s financial strategies and to establish a benchmark for evaluating the
Government’s conduct of financial policy. The statement is also consistent with section
11(1)(a) of the Act.
Strategic Priorities and Financial Policy
In this budget, the Government continues its commitment to the principles of responsible
financial management.
The financial objectives and the key measures below o

Saved page texts to: data/processed/pages.jsonl


Chunking the per-page text into retrieval-friendly slices with overlap

In [46]:
import os, json
from pathlib import Path
from typing import Iterable, List, Tuple, Dict, Any

import fitz                        # <-- PyMuPDF (needed!)
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

SSD_DIR     = Path(r"E:\policy-chatbot")            # <= you can rename this folder if you like :)
PERSIST_DIR = SSD_DIR / "indexes" / "chroma_policy"
HF_CACHE    = SSD_DIR / "hf_cache"

#Actual PDF location 
PDF_PATH = Path(r"E:\Study\WORK\Join Venture Ai\For Task - Policy file.pdf")

# ---------- Knobs ----------
COLLECTION  = "policy"
MAX_CHARS   = 700      # larger chunks => fewer chunks => faster
OVERLAP     = 150
BATCH_SIZE  = 16        # lower if RAM is tight
MAX_PAGES   = None      # set to 20 for a quick test, then None
CLEAR_OLD   = True      # start fresh each run (avoids duplicate IDs)

# ---------- Prepare folders & caches ----------
SSD_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))

# ---------- Quick check ----------
print("Using PDF:", PDF_PATH)
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found at:\n{PDF_PATH}\n"
                            "Check the path/spelling and that the file extension is .pdf.")

# ---------- Fast read (light clean) ----------
def read_pdf_pages(pdf_path: Path) -> Iterable[Tuple[int, str]]:
    doc = fitz.open(str(pdf_path))
    try:
        for i in range(len(doc)):
            text = doc[i].get_text("text") or ""
            text = " ".join(text.split())   # quick whitespace normalize
            yield i + 1, text
    finally:
        doc.close()

# ---------- Fast window chunking ----------
def fast_chunks(page_text: str, page_num: int,
                max_chars: int = MAX_CHARS, overlap: int = OVERLAP):
    t = page_text or ""
    n = len(t)
    if n == 0:
        return
    start, cid = 0, 0
    while start < n:
        end = min(n, start + max_chars)
        chunk = t[start:end].strip()
        if chunk:
            yield {
                "id": f"p{page_num:03d}_c{cid:03d}",
                "text": chunk,
                "metadata": {"page": page_num, "char_start": start, "char_end": end},
            }
            cid += 1
        if end >= n:
            break
        start = max(0, end - overlap)

# ---------- Init Chroma  ----------
client = PersistentClient(path=str(PERSIST_DIR))
if CLEAR_OLD:
    try:
        client.delete_collection(name=COLLECTION)  # use named arg for safety
    except Exception:
        pass
collection = client.get_or_create_collection(name=COLLECTION)

# ---------- Embedder (local, no API key) ----------
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBED_MODEL)

def embed_texts(texts: List[str]) -> List[List[float]]:
    return embedder.encode(texts, normalize_embeddings=True).tolist()

# ---------- Stream pages → chunks → batch-embed → add to Chroma ----------
buf_ids, buf_txts, buf_meta = [], [], []

def flush():
    if not buf_ids:
        return
    vecs = embed_texts(buf_txts)
    collection.add(ids=buf_ids, documents=buf_txts, metadatas=buf_meta, embeddings=vecs)
    buf_ids.clear(); buf_txts.clear(); buf_meta.clear()

total_chunks = 0
for idx, (pg, txt) in enumerate(read_pdf_pages(PDF_PATH), start=1):
    if MAX_PAGES is not None and idx > MAX_PAGES:
        break
    for ch in fast_chunks(txt, pg):
        buf_ids.append(ch["id"]); buf_txts.append(ch["text"]); buf_meta.append(ch["metadata"])
        if len(buf_ids) >= BATCH_SIZE:
            flush()
            total_chunks += BATCH_SIZE

flush()

print(f"Ingest complete. Collection '{COLLECTION}' at: {PERSIST_DIR}")
print("Chunks indexed:", collection.count())

# ---------- Smoke test retrieval ----------
def retrieve(q: str, k: int = 3):
    qvec = embed_texts([q])
    res = collection.query(query_embeddings=qvec, n_results=k)
    for cid, doc, meta in zip(res["ids"][0], res["documents"][0], res["metadatas"][0]):
        page = meta.get('page')
        snippet = doc[:220].replace("\n", " ")
        print(f"- {cid}  (p.{page}): {snippet}{'…' if len(doc)>220 else ''}")

print("\nSample retrieval:")
retrieve("What are the key financial objectives?", k=3)


Using PDF: E:\Study\WORK\Join Venture Ai\For Task - Policy file.pdf
Ingest complete. Collection 'policy' at: E:\policy-chatbot\indexes\chroma_policy
Chunks indexed: 24

Sample retrieval:
- p001_c001  (p.1): the Government continues its commitment to the principles of responsible financial management. The financial objectives and the key measures below outline the Government’s financial policy. Strategic priorities, as they …
- p002_c001  (p.2): t. Table 1.2.1 Financial Objectives and Measures Short Term Financial Objectives Long Term Financial Objectives Maintain a Triple A credit rating Maintain Territory infrastructure Maintain low levels of debt Make adequat…
- p006_c000  (p.6): (c) achieving and maintaining levels of Territory net worth to provide a buffer against factors that may impact adversely on levels of Territory net worth in the future; (d) managing prudently the fiscal risks of the Ter…


Reusable retrieval (ChromaDB + bge-small)

In [47]:
from pathlib import Path
from typing import List, Dict, Any, Optional

import chromadb
from chromadb import PersistentClient          # newer Chroma API
from sentence_transformers import SentenceTransformer

# ---- Paths & settings  ----
PERSIST_DIR     = Path(r"E:\policy-chatbot\indexes\chroma_policy")
COLLECTION_NAME = "policy"
EMBED_MODEL     = "BAAI/bge-small-en-v1.5"     # must match the model used for indexing
K_DEFAULT       = 6

# ---- Load embedder (same model as indexing) ----
_embedder = SentenceTransformer(EMBED_MODEL)
def _embed(texts: List[str]) -> List[List[float]]:
    return _embedder.encode(texts, normalize_embeddings=True).tolist()

# ---- Load Chroma persistent collection ----
client = PersistentClient(path=str(PERSIST_DIR))
collection = client.get_or_create_collection(name=COLLECTION_NAME)

def retrieve(
    query: str,
    k: int = K_DEFAULT,
    where: Optional[Dict[str, Any]] = None,          # metadata filter, e.g., {"page": {"$gte": 3}}
    include: List[str] = ["documents", "metadatas", "distances"]  
) -> List[Dict[str, Any]]:
    """
    Return top-k hits with text, metadata (page), id, and distance.
    Note: distances are Chroma's metric-specific values (smaller is typically better).
    """
    qvec = _embed([query])
    res = collection.query(query_embeddings=qvec, n_results=k, where=where, include=include)

    docs   = res.get("documents", [[]])[0]
    metas  = res.get("metadatas", [[]])[0]
    dists  = res.get("distances", [[]])[0] if "distances" in res else [None] * len(docs)

    results = []
    
    for idx, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        page = (meta or {}).get("page")
        snippet = (doc or "")[:260].replace("\n", " ")
        results.append({
            "id": f"result_{idx+1}",  
            "page": page,
            "distance": dist,
            "text": doc,
            "snippet": snippet + ("…" if doc and len(doc) > 260 else "")
        })
    return results

def print_hits(hits: List[Dict[str, Any]]):
    """Pretty-print retrieval results."""
    if not hits:
        print("No results.")
        return
    for i, h in enumerate(hits, 1):
        print(f"{i}. {h['id']}  (p.{h.get('page')})  dist={h.get('distance')}")
        print(f"   {h['snippet']}")
        print("-" * 80)

# ---- Quick smoke tests  ----
hits = retrieve("What are the key financial objectives?", k=5)
print_hits(hits)

# Example with a simple metadata filter (only pages >= 2)
# hits_filtered = retrieve("net worth", k=5, where={"page": {"$gte": 2}})
# print_hits(hits_filtered)


1. result_1  (p.1)  dist=0.38600635528564453
   the Government continues its commitment to the principles of responsible financial management. The financial objectives and the key measures below outline the Government’s financial policy. Strategic priorities, as they relate to the Territory’s Budget, are su…
--------------------------------------------------------------------------------
2. result_2  (p.2)  dist=0.4388740062713623
   t. Table 1.2.1 Financial Objectives and Measures Short Term Financial Objectives Long Term Financial Objectives Maintain a Triple A credit rating Maintain Territory infrastructure Maintain low levels of debt Make adequate provision for long-term liabilities St…
--------------------------------------------------------------------------------
3. result_3  (p.6)  dist=0.46510636806488037
   (c) achieving and maintaining levels of Territory net worth to provide a buffer against factors that may impact adversely on levels of Territory net worth in the future; (

RAG Answer Generation (offline extractive, with page citations)

In [48]:
import re, numpy as np
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
try:
    _embedder
except NameError:
    _embedder = SentenceTransformer(EMBED_MODEL)

def _embed(texts: List[str]) -> np.ndarray:
    vecs = _embedder.encode(texts, normalize_embeddings=True)
    return np.asarray(vecs, dtype=np.float32)

_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(0-9])")

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text: return []
    parts = _SENT_SPLIT_RE.split(text)
    out = []
    for s in parts:
        s = re.sub(r"\s+", " ", s).strip()
        if 30 <= len(s) <= 320:
            out.append(s)
    return out

def compact_pages(pages: List[int]) -> str:
    pages = sorted({p for p in pages if isinstance(p, int)})
    if not pages: return ""
    ranges, start, prev = [], pages[0], pages[0]
    for p in pages[1:]:
        if p == prev + 1: prev = p
        else: ranges.append((start, prev)); start = prev = p
    ranges.append((start, prev))
    return ", ".join(f"{a}" if a==b else f"{a}–{b}" for a,b in ranges)

def _classify_question(q: str) -> str:
    ql = q.lower().strip()
    if any(w in ql for w in ["how much","how many","percent","%","value","figure","amount","2004","2005","table"]):
        return "numeric"
    if ql.startswith(("what are","list","which are","name the","enumerate","objectives","principles")):
        return "list"
    if ql.startswith(("what is","define","explain","summarize","summarise")):
        return "definition"
    return "general"

def _is_tabley(s: str) -> bool:
    digits = sum(c.isdigit() for c in s)
    ratio = digits / max(1, len(s))
    many_seps = s.count(",") + s.count("$") + s.count("%")
    long_tokens = len([t for t in s.split() if len(t) > 18])
    return ratio > 0.20 or many_seps >= 3 or long_tokens >= 2

def _mmr_select(qv: np.ndarray, cand: np.ndarray, k: int, lam: float = 0.65) -> List[int]:
    selected = []
    if cand.size == 0: return selected
    sim_q = cand @ qv
    best = int(np.argmax(sim_q)); selected.append(best)
    remaining = [i for i in range(len(cand)) if i != best]
    for _ in range(k-1):
        if not remaining: break
        max_div = []
        for j in remaining:
            max_div.append(float(np.max(cand[j] @ cand[selected].T)))
        mmr = lam * sim_q[remaining] - (1 - lam) * np.asarray(max_div)
        j_idx = int(np.argmax(mmr))
        selected.append(remaining[j_idx])
        remaining.pop(j_idx)
    return selected

def rag_answer_v2(query: str, k_chunks: int = 10, k_sentences: int = 5) -> Dict[str, Any]:
    qtype = _classify_question(query)
    where = None
    if qtype in ("definition","general"):
        where = {"page": {"$lte": 4}}  

    hits = retrieve(query, k=k_chunks, where=where)
    if not hits:
        return {"answer":"I don’t have enough information in the document to answer that.","sentences":[],"sources":[]}

    candidates, pages_seen, raw_blobs = [], [], []
    for h in hits:
        pg = h.get("page")
        pages_seen.append(pg)
        raw = h.get("text","")
        raw_blobs.append(raw)
        for s in split_sentences(raw):
            candidates.append((s, pg))

    if not candidates:
        src = compact_pages([p for p in pages_seen if p])
        fallback = " ".join(h["text"][:300].replace("\n"," ") for h in hits[:2])
        return {"answer": fallback + (f"\n\nSources: p. {src}" if src else ""), "sentences": [], "sources": pages_seen}

    # Filtered number-heavy lines except for numeric Qs
    if qtype != "numeric":
        candidates = [(s,p) for (s,p) in candidates if not _is_tabley(s)]
        if not candidates:  # fallback if we filtered too hard
            candidates = [(split_sentences(h.get("text",""))[0], h.get("page")) for h in hits if split_sentences(h.get("text",""))]

    sents = [s for s,_ in candidates]
    pgs   = [p for _,p in candidates]
    qv    = _embed([query])[0]
    sv    = _embed(sents)

    chosen_idx = _mmr_select(qv, sv, k=min(k_sentences, len(sents)), lam=0.7)
    chosen = [(sents[i], pgs[i], float(sv[i] @ qv)) for i in chosen_idx]

    # Format based on question type
    pages_used = [p for _,p,_ in chosen if p]
    pages_str  = compact_pages(pages_used or [p for p in pages_seen if p])

    if qtype == "list":
        bullets = []
        for t in raw_blobs:
            for line in t.splitlines():
                L = line.strip(" •–-")
                if 10 < len(L) < 140 and L[0].isalpha() and not _is_tabley(L):
                    if any(w in L.lower() for w in ["maintain","increase","prudent","stable","disclosure","manage"]):
                        bullets.append(L)
        bullets = list(dict.fromkeys(bullets))
        if not bullets:
            bullets = [s for s,_,_ in chosen]
        bullets = bullets[:6]
        ans = "Here are the key points:\n" + "\n".join(f"• {b}" for b in bullets)
    elif qtype == "definition":
        chosen.sort(key=lambda t: -t[2])
        main = chosen[0]
        support = [c for c in chosen[1:3]]
        parts = [main[0] + (f" (p. {main[1]})" if main[1] else "")]
        for s,p,_ in support:
            parts.append(s + (f" (p. {p})" if p else ""))
        ans = " ".join(parts)
    else:
        chosen.sort(key=lambda t: (t[1] if t[1] else 10**9, -t[2]))
        parts = []
        for s,p,_ in chosen[:4]:
            parts.append(s + (f" (p. {p})" if p else ""))
        ans = " ".join(parts)

    if pages_str:
        ans += f"\n\nSources: p. {pages_str}"
    return {
        "answer": ans,
        "sentences": [{"text": s, "page": p, "similarity": round(sim,3)} for s,p,sim in chosen],
        "sources": pages_used,
    }

def answer_with_memory_v2(query: str, memory: ConversationMemory, k: int = 8, show_debug: bool = False) -> str:
    aug = memory.augment_query(query)
    res = rag_answer_v2(aug, k_chunks=k, k_sentences=5)
    memory.add_turn(user=query, answer=res.get("answer",""), sources=res.get("sources",[]))
    if show_debug and res.get("sentences"):
        print("Chosen sentences:")
        for s in res["sentences"]:
            print(f"- (p.{s['page']}) sim={s['similarity']}: {s['text'][:120]}{'…' if len(s['text'])>120 else ''}")
        print("Sources:", compact_pages(res.get("sources",[])))
    return res.get("answer","I don’t have enough information in the document to answer that.")


Conversation memory (rolling buffer + context-augmented queries)

In [51]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from collections import Counter
import re

try:
    compact_pages
except NameError:
    def compact_pages(pages: List[int]) -> str:
        pages = sorted({p for p in pages if isinstance(p, int)})
        if not pages: return ""
        ranges, start, prev = [], pages[0], pages[0]
        for p in pages[1:]:
            if p == prev + 1:
                prev = p
            else:
                ranges.append((start, prev))
                start = prev = p
        ranges.append((start, prev))
        return ", ".join(f"{a}" if a == b else f"{a}–{b}" for a, b in ranges)

# Simple stopword list (kept tiny to avoid extra deps)
_STOP = {
    "the","a","an","and","or","to","of","in","on","for","with","by","from","as","is","are","was","were","be","being",
    "been","at","that","this","it","its","into","over","under","about","what","which","who","whom","how","why","when",
    "do","does","did","doing","done","than","then","there","their","them","they","we","you","your","our"
}

def _keywords(text: str, top_k: int = 10) -> List[str]:
    toks = re.findall(r"[A-Za-z0-9%.-]+", text.lower())
    toks = [t for t in toks if t not in _STOP and len(t) > 2]
    cnt = Counter(toks)
    return [w for w, _ in cnt.most_common(top_k)]

@dataclass
class Turn:
    user: str
    answer: str
    sources: List[int] = field(default_factory=list)

class ConversationMemory:
    def __init__(self, max_turns: int = 5, max_summary_chars: int = 400):
        self.max_turns = max_turns
        self.max_summary_chars = max_summary_chars
        self.turns: List[Turn] = []

    def add_turn(self, user: str, answer: str, sources: List[int]):
        self.turns.append(Turn(user=user.strip(), answer=answer.strip(), sources=sources or []))
        self.turns = self.turns[-self.max_turns:]  # keep last N

    def summary(self) -> str:
        if not self.turns:
            return ""
        # Last 2 user prompts (most recent first)
        last_qs = [t.user for t in self.turns[-2:]][::-1]
        qs_part = " | ".join(last_qs)
        # Aggregate sources across recent answers
        pages = []
        for t in self.turns:
            pages.extend(t.sources or [])
        src_part = compact_pages(pages)
        # Keywords from recent user prompts
        kw = _keywords(" ".join(t.user for t in self.turns), top_k=8)
        summary = f"Recent topics: {', '.join(kw)}."
        if qs_part:
            summary += f" Last questions: {qs_part}."
        if src_part:
            summary += f" Recent cited pages: {src_part}."
        # Trim if too long
        return (summary[: self.max_summary_chars] + "…") if len(summary) > self.max_summary_chars else summary

    def augment_query(self, query: str) -> str:
        s = self.summary()
        if not s:
            return query
        # Keep augmentation simple—this biases retrieval without polluting the final answer
        return f"{query}\n\n[Conversation context] {s}"

    def reset(self):
        self.turns.clear()

# --- Memory-aware answer wrapper ---
def answer_with_memory(query: str, memory: ConversationMemory, k: int = 6, show_debug: bool = False) -> str:
    aug_query = memory.augment_query(query)
    res = rag_answer_v2(aug_query, k=k)  
    memory.add_turn(user=query, answer=res.get("answer", ""), sources=res.get("sources", []))
    if show_debug:
        print("Conversation summary:", memory.summary())
        if res.get("sentences"):
            print("Chosen sentences:")
            for s in res["sentences"]:
                print(f"- (p.{s['page']}) sim={s['similarity']}: {s['text'][:120]}{'…' if len(s['text'])>120 else ''}")
        if res.get("sources"):
            print("Sources pages:", compact_pages(res["sources"]))
        print()
    return res.get("answer", "I don’t have enough information in the document to answer that.")

# --- Quick usage ---
mem = ConversationMemory(max_turns=5)
print(answer_with_memory("What are the key financial policy objectives?", mem, k=6, show_debug=True))
print(answer_with_memory("And what about net worth or debt targets?", mem, k=6, show_debug=True))

# Helper to inspect memory or clear it
# print("Summary:", mem.summary())
# mem.reset()


TypeError: rag_answer_v2() got an unexpected keyword argument 'k'

In-notebook chat loop (with conversation memory + citations)

In [ ]:
import html
import ipywidgets as W
from IPython.display import display, HTML

mem = ConversationMemory(max_turns=5)

k_slider  = W.IntSlider(value=8, min=3, max=12, step=1, description="Top-k", continuous_update=False)
inp       = W.Text(description="You:", placeholder="Ask a question… (press Enter)")
send_btn  = W.Button(description="Send", button_style="success")
clear_btn = W.Button(description="Clear Memory", button_style="warning")
help_btn  = W.Button(description="Help")

# key change: taller box + wrap; hided horizontal scroll
out = W.Output(layout=W.Layout(
    width="100%", height="560px",
    overflow_y="auto", overflow_x="hidden",
    border="1px solid #ccc"
))

_state = {"busy": False}

def _show_html(text, prefix=None):
    safe = html.escape(text)
    if prefix:
        block = f"<div style='white-space:pre-wrap; line-height:1.45;'><b>{prefix}</b> {safe}</div>"
    else:
        block = f"<div style='white-space:pre-wrap; line-height:1.45;'>{safe}</div>"
    with out:
        display(HTML(block))

def _rule():
    with out:
        display(HTML("<hr style='border:none;border-top:1px solid #ddd;margin:8px 0;'/>"))

def _help(_=None):
    tips = (
        "- Use the slider to change retrieval Top-k (context size).\n"
        "- Click 'Clear Memory' to forget the last turns.\n"
        "- Press Enter in the input box or click 'Send' to ask.\n"
        "- Answers include page citations like (p. X) and a 'Sources: p. …' line."
    )
    _show_html(tips)
    _rule()

def _submit(_=None):
    if _state["busy"]:
        return
    q = (inp.value or "").strip()
    if not q:
        return
    _state["busy"] = True
    inp.value = ""
    _show_html(q, prefix="You:")
    try:
        ans = answer_with_memory_v2(q, mem, k=k_slider.value, show_debug=False)
    except Exception as e:
        ans = f"[Error] {e}"
    _show_html(ans, prefix="Bot:")
    _rule()
    _state["busy"] = False

def _clear(_):
    mem.reset()
    _show_html("[Memory cleared]")
    _rule()

# wire events (submit only on Enter or Send)
inp.on_submit(_submit)
send_btn.on_click(_submit)
clear_btn.on_click(_clear)
help_btn.on_click(_help)

header = W.HBox([k_slider, send_btn, clear_btn, help_btn])
display(header, inp, out)
_help()


C:\Users\Asus\AppData\Local\Temp\ipykernel_11584\2154444505.py:75: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  inp.on_submit(_submit)


Text(value='', description='You:', placeholder='Ask a question… (press Enter)')

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…